In [19]:
import casadi as cs
import numpy as np
from scipy.optimize import brentq
from time import perf_counter
from matplotlib import pyplot as plt
import pickle

In [20]:
class DecisionLayout():

    def __init__(self,modes,N,dim):
        self.modes = tuple(modes)   # ('s','f','s')
        self.N = N                  # usage: N["f"] = N_f
        self.dim = dim              # usage: dim["f"] = 6

        k = 0
        self.x_slice = []
        for m in modes:
            n = dim[m] * N[m]
            self.x_slice.append(slice(k,k+n))
            k += n

        # k = sum_q(N_q * dim_q)
        self.u_slice = []
        for m in modes:
            n = N[m] - 1 # number of control inputs in mode m
            self.u_slice.append(slice(k,k+n))
            k += n

        self.t_slice = slice(k, k+len(modes))
        k += len(modes)

        self.E_index = k
        self.phi_index = k + 1
        self.nz = k + 2

    def pack(self,x,u,t,E,phi):
        parts = [cs.vec(X) for X in x]
        parts += [cs.vec(U) for U in u]
        parts += [cs.vec(t), E, phi]
        return cs.vertcat(*parts)

    def unpack(self,z):
        x = [
            cs.reshape(z[s], self.dim[m], self.N[m])
            for m, s in zip(self.modes, self.x_slice)
        ]

        u = [
            z[s]
            for s in self.u_slice
        ]

        return {
            "modes": self.modes,
            "x": x,
            "u": u,
            "t": z[self.t_slice],
            "E": z[self.E_index],
            "phi": z[self.phi_index],
        }



In [21]:
data = pickle.load(open('../data/dataset_2/dataset_21.pkl','rb'))
K = data["K"]
W = data["W"]
Nf = data["Nf"]
Ns = data["Ns"]
locomotion_NMM = np.array(data["locomotion"][2])
switching_NMM = np.array(data["switch"][2])
Emin = max(min(locomotion_NMM[:,-1]),min(switching_NMM[:,-1]))
Emax = min(max(locomotion_NMM[:,-1]),max(switching_NMM[:,-1]))

In [22]:
def RK4(f,x,h,u):
    k1 = f(x,u)
    k2 = f(x+0.5*h*k1,u)
    k3 = f(x+0.5*h*k2,u)
    k4 = f(x+h*k3,u)
    x_next = x + h/6 * (k1 + 2*k2 + 2*k3 +k4)
    return x_next

In [23]:
xs = cs.SX.sym("xs", 4)
xf = cs.SX.sym("xf", 6)
u = cs.SX.sym("u")
h = cs.SX.sym("h")

l = cs.sqrt(xs[0] ** 2 + xs[1] ** 2)
dxs = cs.vertcat(
    xs[2:], 
    (K * (1 - l) + u) * xs[0] / l, 
    (K * (1 - l) + u) * xs[1] / l - 1
)
dxf = cs.vertcat(
    xf[3:], 
    0, 
    -1, 
    -(W**2) * xf[2] + u
)
xfs = cs.vertcat(
    xs[:2],
    cs.atan2(-xs[0],xs[1]),
    xs[2:],
    (xs[0] * xs[3] - xs[1] * xs[2]) / l**2
)
xsf = cs.vertcat(
    -cs.sin(xf[2]),
    xf[[1,3,4]]
)
e_TD = cs.cos(xf[2]) - xf[1]
e_LO = l - 1
de_TD = -cs.sin(xf[2])*xf[5] - xf[4]
de_LO = xs[0]*xs[2] + xs[1]*xs[3]


In [24]:
f =  {
    "f": cs.Function("ff", [xf, u], [dxf]), 
    "s": cs.Function("fs", [xs, u], [dxs])
}
rk4 = {
    "f": cs.Function("RK4f", [xf,h,u],[RK4(f["f"],xf,h,u)]),
    "s": cs.Function("RK4s", [xs,h,u],[RK4(f["s"],xs,h,u)])
}
convert = {
    "s": cs.Function("to_flight", [xs],[xfs]),
    "f": cs.Function("to_stance", [xf],[xsf])
}
guard = {
    "s" : cs.Function("guards", [xs], [e_LO]),
    "f" : cs.Function("guardf", [xf], [e_TD])
}
guard_rate = {
    "s" : cs.Function("guards", [xs], [de_LO]),
    "f" : cs.Function("guardf", [xf], [de_TD])
}
next_mode = {
    "f":"s", 
    "s":"f"
}
dim = {
    "f":6, 
    "s":4
}
N = {
    "f":15, 
    "s":20
}

In [25]:
def energy_sample(M,grid):
    idx = np.argsort(M[:,-1])
    M = M[idx,:]
    E = M[:,-1]
    return np.column_stack([np.interp(grid, E, M[:, c]) for c in range(M.shape[1])])

E_grid = np.linspace(Emin, Emax, 20)
locomotion_NMM_sampled = energy_sample(locomotion_NMM,E_grid)
switching_NMM_sampled = energy_sample(switching_NMM,E_grid)

In [26]:
def integrate_until_event(x0,mode,N_max=1000,h=1e-2):
    x = x0
    t = 0.0
    xh = []
    th = []
    armed = False
    
    for i in range(N_max):

        xh.append(x)
        th.append(t)
        x_next = rk4[mode](x,h,0.0)

        g0 = guard[mode](x)
        g1 = guard[mode](x_next)

        if (not armed) and g1 < 0 and g0<0:
            armed = True

        if g0 * g1 < 0 and armed:
            def residual(tau):
                x_tau = rk4[mode](x,tau,0.0)
                return guard[mode](x_tau)
            
            tau_event = brentq(residual,0.0,h,xtol=1e-6,rtol=1e-6)
            x_event = rk4[mode](x,tau_event,0.0)
            xh.append(x_event)
            th.append(t+tau_event)

            return np.column_stack(th),np.column_stack(xh)

        t += h
        x = x_next

In [27]:
def trajectory_sample(u,grid):
    x0 = u[:6]
    th,xh = integrate_until_event(x0,"f")
    tTD,xTD = th[0,-1],convert["f"](xh[:,-1])
    th_s,xh_s = integrate_until_event(xTD,"s")
    tLO,xLO = th_s[0,-1],convert["s"](xh_s[:,-1])
    th_f,xh_f = integrate_until_event(xLO,"f")
    ts_norm = th_s / th_s[0,-1]
    tf_norm = th_f / th_f[0,-1] 

    traj_s = np.column_stack([
        np.interp(grid, ts_norm.squeeze(), xh_s[r,:]) for r in range(xh_s.shape[0])
    ]+[np.interp(grid, ts_norm.squeeze(), th_s.squeeze())])
    traj_f = np.column_stack([
        np.interp(grid, tf_norm.squeeze(), xh_f[r,:]) for r in range(xh_f.shape[0])
    ]+[np.interp(grid, tf_norm.squeeze(), th_f.squeeze())])
    # traj_s = convert["s"](traj_s.T).T

    return traj_s,traj_f

def surface(M,grid):
    N = M.shape[0]
    surf_s,surf_f = np.zeros((N,N,5)),np.zeros((N,N,7))
    for i in range(N):
        ts,tf = trajectory_sample(M[i],grid)
        surf_s[:,i,:] = ts
        surf_f[:,i,:] = tf
    return surf_s,surf_f

In [28]:
phi_grid = np.linspace(0,1,20)
ss_loc,sf_loc = surface(locomotion_NMM_sampled,phi_grid)
ss_swi,sf_swi = surface(switching_NMM_sampled,phi_grid)

values_sloc = ss_loc.ravel(order="C")
values_floc = sf_loc.ravel(order="C")
values_sswi = ss_swi.ravel(order="C")
values_fswi = sf_swi.ravel(order="C")
LUT_sloc = cs.interpolant('LUT_sloc', 'bspline',[E_grid,phi_grid],values_sloc)
LUT_floc = cs.interpolant('LUT_floc', 'bspline',[E_grid,phi_grid],values_floc)
LUT_sswi = cs.interpolant('LUT_sswi', 'bspline',[E_grid,phi_grid],values_sswi)
LUT_fswi = cs.interpolant('LUT_fswi', 'bspline',[E_grid,phi_grid],values_fswi)

In [29]:
def make_problem(layout,target_LUT,rho=1e-6,rho_slack=1e3):

    z = cs.MX.sym("z",layout.nz)
    variables = layout.unpack(z)

    x = variables["x"]
    u = variables["u"]
    t = variables["t"]
    E = variables["E"]
    phi = variables["phi"]
    h = [t[i]/(layout.N[m]-1) for i,m in enumerate(layout.modes)]
    target = target_LUT(cs.vertcat(E,phi))

    nx0 = layout.dim[layout.modes[0]]
    p = cs.MX.sym("p", nx0 + 1)

    x0 = p[:nx0]
    T = p[nx0]

    q_list = []
    for i,m in enumerate(layout.modes):
        q_list.append(cs.sqrt(2*h[i])*u[i])
    q = cs.vertcat(*q_list)
    J = 0.5 * q.T @ q
    J_z = cs.gradient(J,z)
    Q = cs.jacobian(q,z)
    H = Q.T @ Q 
    H_regul = H + rho * cs.DM.eye(layout.nz)

    # initial constraint
    c_list = [x[0][:,0] - x0]

    # dynamical constraint
    for i,m in enumerate(layout.modes):
        # i index of mode m
        for k in range(layout.N[m]-1):
            # k discrete time
            c_list.append(x[i][:,k+1] - rk4[m](x[i][:,k],h[i],u[i][k]))

    # event crossing 
    for i in range(len(layout.modes)-1):
        m = layout.modes[i]
        c_list.append(x[i+1][:,0] - convert[m](x[i][:,-1])) # reset
        c_list.append(guard[m](x[i][:,-1]))                 # guard

    # terminal constraint
    idx_t = sum(ci.numel() for ci in c_list)
    if layout.modes[-1] == "f":
        r_t = target[1:-1] - x[-1][1:,-1]
    if layout.modes[-1] == "s":
        r_t = target[:-1] - x[-1][:,-1]
    c_list.append(r_t)


    # horizon
    c_list.append(cs.sum(t) - T)

    # equality constraint
    c = cs.vertcat(*c_list)
    C = cs.jacobian(c,z)
    n_e = c.shape[0]
    n_t = r_t.numel()
    slice_t = slice(idx_t, idx_t + n_t)

    # slack variables
    S = cs.DM(cs.Sparsity(n_e, 2*n_t))
    S[slice_t, :n_t] = -cs.DM.eye(n_t)
    S[slice_t, n_t:] = cs.DM.eye(n_t)

    g_list = []
    # guard rate
    for i in range(len(layout.modes)-1):
        m = layout.modes[i]
        g_list.append(guard_rate[m](x[i][:,-1]))

    # guard inside last mode
    for k in range(1,layout.N[layout.modes[-1]]):
        g_list.append(guard[layout.modes[-1]](x[-1][:,k]))

    # inequality constraint
    g = cs.vertcat(*g_list)
    G = cs.jacobian(g,z)
    n_i = g.shape[0]

    # sparsity
    grad_qp = cs.vertcat(J_z, rho_slack*cs.DM.ones(2*n_t))
    A_qp = cs.vertcat(
        cs.horzcat(C,S),
        cs.horzcat(G,cs.DM(cs.Sparsity(n_i,2*n_t)))
    )
    H_qp = cs.diagcat(
        H_regul,
        1e-10 * cs.DM.eye(2*n_t)
    )
    qp_structure = {
        "h": H_qp.sparsity(),
        "a": A_qp.sparsity()
    }


    values = cs.Function("values",[z,p],[grad_qp,c,g])
    linearize = cs.Function("linearize",[z,p],[grad_qp,H_qp,A_qp,c,g],{ "jit": True,"compiler": "shell"})

    return values,linearize,qp_structure,n_t


In [30]:
def evaluate(z,values,g_min,g_max,rho=1e3):
    J,c,g = values(z)
    violation = np.maximum(g_min-g,0) + np.maximum(g-g_max,0)
    merit = J + rho*(np.sum(np.abs(c)) + np.sum(violation))
    return merit,J,np.max(np.abs(c)),np.max(violation)

In [31]:
def make_bounds(layout, x_bounds, u_bounds, t_bounds, E_bounds, phi_bounds):
    z_min = np.full(layout.nz, -np.inf)
    z_max = np.full(layout.nz, np.inf)

    # state bound
    for i,mode in enumerate(layout.modes):
        nx = layout.dim[mode]
        nk = layout.N[mode]
        x_min = np.full((nx, nk), -np.inf)
        x_max = np.full((nx, nk),  np.inf)
        for idx,bounds in x_bounds.get(mode, {}).items():
            x_min[idx,:] = bounds[0]
            x_max[idx,:] = bounds[1]
        z_min[layout.x_slice[i]] = x_min.reshape(-1,order="F")
        z_max[layout.x_slice[i]] = x_max.reshape(-1,order="F")

    # control bound
    for i,mode in enumerate(layout.modes):
        u_min, u_max = u_bounds[mode]
        z_min[layout.u_slice[i]] = u_min
        z_max[layout.u_slice[i]] = u_max

    # duration bound 
    for i,mode in enumerate(layout.modes):
        t_min, t_max = t_bounds[mode]
        idx = layout.t_slice.start + i
        z_min[idx] = t_min
        z_max[idx] = t_max

    # NMM parameters bounds
    z_min[layout.E_index],z_max[layout.E_index] = E_bounds
    z_min[layout.phi_index],z_max[layout.phi_index] = phi_bounds

    return z_min,z_max

def make_qp_bounds(z_bar,z_min,z_max,delta=None):
    z_bar = np.asarray(z_bar).squeeze()

    lbz = z_min - z_bar
    ubz = z_max - z_bar

    if delta is not None:
        lbz = np.maximum(lbz, -delta)
        ubz = np.minimum(ubz,  delta)

    return lbz, ubz


In [32]:
def qp_data(z,p,linearize,n_events,n_final_guard,n_terminal):

    grad_qp,H_qp,A_qp,c,g = linearize(z,p)
    g_min = cs.vertcat(
        1e-3 * cs.DM.ones(n_events),
        -cs.inf * cs.DM.ones(n_final_guard),
    )

    g_max = cs.vertcat(
        cs.inf * cs.DM.ones(n_events),
        -1e-3 * cs.DM.ones(n_final_guard),
    )
    lba = cs.vertcat(-c,g_min - g)
    uba = cs.vertcat(-c,g_max - g)
    x_bounds = {
        "f" : {1: (0.0, np.inf)}, 
        "s" : {}
    }
    u_bounds = {
        "f" : (-1,1), 
        "s" : (-1,1)
    }
    t_bounds = {
        "f": (1e-2, 10.0),
        "s": (1e-2, 10.0),
    }
    E_bounds = (E_inf,E_sup)
    phi_bounds = (0.01,0.99)
    z_min, z_max = make_bounds(layout, x_bounds, u_bounds, t_bounds, E_bounds, phi_bounds)
    delta = np.full(layout.nz, np.inf)

    for state_slice in layout.x_slice:
        delta[state_slice] = 0.2

    for control_slice in layout.u_slice:
        delta[control_slice] = 0.5

    delta[layout.t_slice] = 0.1
    delta[layout.E_index] = 0.1
    delta[layout.phi_index] = 0.2

    lbz, ubz = make_qp_bounds(z_bar,z_min,z_max,delta)
    lbw = cs.vertcat(lbz,cs.DM.zeros(2*n_terminal))
    ubw = cs.vertcat(ubz,cs.inf*cs.DM.ones(2*n_terminal))

    return dict(h=H_qp, g=grad_qp, a=A_qp, lbx=lbw, ubx=ubw, lba=lba, uba=uba)



In [33]:
modes = ("s","f","s")
layout = DecisionLayout(modes,N,dim)
values,linearize,qp_structure,n_t = make_problem(layout,LUT_sswi)

qp_solver = cs.conic(
    "qp_solver",
    "proxqp",
    qp_structure,
    {
        "print_time": False,
        "error_on_fail": False,
    },
)

In [34]:
def initialize_mode(m,LUT,N,E0,phi_end=1.0):
    grid = np.linspace(0,phi_end,N)
    x_init = np.zeros((dim[m],N))
    for i in range(N):
        x_init[:,i] = np.array(LUT([E0,grid[i]])[:-1]).squeeze()
    return x_init


In [35]:
E0 = 4
E_inf = max(Emin,E0-0.2)
E_sup = min(Emax,E0+0.2)

ts,tf = LUT_sloc([E0,1])[-1],LUT_floc([E0,1])[-1]
T = float(ts + tf + 0.5*ts)
t_init = cs.vertcat(ts,tf,0.5*ts)
xs_init = initialize_mode("s",LUT_sloc,N["s"],E0) ## TODO use rk4 instead
xf_init = initialize_mode("f",LUT_floc,N["f"],E0)
x_init = [xs_init, xf_init, initialize_mode("s",LUT_sloc,N["s"],E0,0.5)]
u_init = [np.zeros(N[m]-1) for m in modes]
E_init = E0
phi_init = 0.5

z_bar = layout.pack(x_init,u_init,t_init,E_init,phi_init)
p = cs.vertcat(xs_init[:,0],T)


In [44]:
n_events = len(layout.modes)-1
n_final_guard =  layout.N[layout.modes[-1]] - 1
start = perf_counter()
result = qp_solver(**qp_data(z_bar,p,linearize,n_events,n_final_guard,n_t))
print(perf_counter() - start)

delta_z = result["x"]
lambda_a = result["lam_a"]
lambda_x = result["lam_x"]

success = qp_solver.stats()["success"]

0.02474935501231812
